In [1]:
!pip install -q opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 98.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.15.3 which is incompatible.
mkl-umath 0.1.1 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
mkl-random 1.2.4 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
mkl-fft 1.3.8 requires numpy<1.27.0,>=1.26.4, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
da

In [2]:
!pip install -U numpy opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.5 MB/s eta 0:00:00


In [1]:
import cv2
import numpy as np
import os
import glob
import random
from tqdm.notebook import tqdm

In [2]:
ffhq_data = "/kaggle/input/faces-dataset-small"
output_dir = "/kaggle/working/processed_ffhq_hed_augmented"

img_size = 512 
hed_prototxt = "/kaggle/input/hed-model/other/default/1/deploy.prototxt"
hed_caffemodel = "/kaggle/input/hed-model/other/default/1/hed_pretrained_bsds.caffemodel"

train_ratio = 0.9  
eval_ratio = 0.05  

# == DATA AUGMENTATION ==

# (1)
# augment_prob = 0.7 
# blur_kernel = 5 
# noise = 0.05
# distortion = 5

# (1)
augment_prob = 0.95
blur_kernel = 9
noise = 0.1
distortion = 10

# (2)
erosion_prob = 1
erosion_size = 6
dropout_prob = 1     
dropout_holes = 65536
dropout_min_size = 1
dropout_max_size = 3     
dropout_fill_value = 255 

In [3]:
net = cv2.dnn.readNetFromCaffe(hed_prototxt, hed_caffemodel)

In [5]:
# def augment_sketch(sketch_image):

#     augmented_sketch = sketch_image.copy()
    
#     # Gaussian Blur 
#     if random.random() < 0.3: 
#         kernel_size = random.choice([k for k in range(3, blur_kernel + 1) if k % 2 != 0])
#         if kernel_size: 
#             augmented_sketch = cv2.GaussianBlur(augmented_sketch, (kernel_size, kernel_size), 0)
    
#     if random.random() < 0.4: 
#         mean = 0
#         std_dev = random.uniform(0.01, noise) * 255 
#         gaussian_noise = np.random.normal(mean, std_dev, augmented_sketch.shape)
#         sketch_float = augmented_sketch.astype(np.float32)
#         augmented_float = sketch_float + gaussian_noise
#         augmented_float = np.clip(augmented_float, 0, 255)
#         augmented_sketch = augmented_float.astype(np.uint8)

#     if random.random() < 0.2: 
#         rows, cols, _ = augmented_sketch.shape
        
#         pts1 = np.float32([[50, 50], [cols - 50, 50], [50, rows - 50]])
        
#         pts2 = np.float32([[50 + random.randint(-distortion, distortion), 
#                             50 + random.randint(-distortion, distortion)],
#                            [cols - 50 + random.randint(-distortion, distortion), 
#                             50 + random.randint(-distortion, distortion)],
#                            [50 + random.randint(-distortion, distortion), 
#                             rows - 50 + random.randint(-distortion, distortion)]])
        
#         M = cv2.getAffineTransform(pts1, pts2)
#         augmented_sketch = cv2.warpAffine(augmented_sketch, M, (cols, rows), borderValue=(255, 255, 255)) 
        
#     return augmented_sketch

In [4]:
def augment_sketch(sketch_image):
    augmented_sketch = sketch_image.copy()

    if random.random() > augment_prob:
        return augmented_sketch

    if random.random() < erosion_prob:
        kernel = np.ones((erosion_size, erosion_size), np.uint8)
        augmented_sketch = cv2.erode(augmented_sketch, kernel, iterations=1)

    if random.random() < dropout_prob:
        rows, cols, _ = augmented_sketch.shape
        
        num_holes = random.randint(int(dropout_holes * 0.8), dropout_holes)
        
        for _ in range(num_holes):
            hole_w = random.randint(dropout_min_size, dropout_max_size) 
            hole_h = random.randint(dropout_min_size, dropout_max_size)
            
            x1 = random.randint(0, cols - hole_w)
            y1 = random.randint(0, rows - hole_h)
            x2 = x1 + hole_w
            y2 = y1 + hole_h
            
            fill_value = dropout_fill_value
            augmented_sketch[y1:y2, x1:x2] = (fill_value, fill_value, fill_value)
            
    return augmented_sketch

In [5]:
def apply_hed(image, net, target_size=512, apply_augmentation=False):
    (h, w) = image.shape[:2]
    
    mean_pixel_values = (104.00698793, 116.66876762, 122.67891434)
    blob = cv2.dnn.blobFromImage(image, 
                                 scalefactor=1.0, 
                                 size=(w, h), 
                                 mean=mean_pixel_values,
                                 swapRB=False, 
                                 crop=False)
    
    net.setInput(blob)
    hed_output = net.forward()
    hed_output = hed_output[0, 0] 
    
    hed_output = cv2.resize(hed_output, (w, h))
    
    hed_output = cv2.normalize(hed_output, None, 0, 255, cv2.NORM_MINMAX)
    hed_output = hed_output.astype("uint8")
    
    hed_output = 255 - hed_output
    hed_output_rgb = cv2.cvtColor(hed_output, cv2.COLOR_GRAY2BGR)
    
    if apply_augmentation and random.random() < augment_prob:
        hed_output_rgb = augment_sketch(hed_output_rgb)
    
    return hed_output_rgb

In [6]:
all_image_paths = sorted(glob.glob(f"{ffhq_data}/**/*.png", recursive=True))

print(len(all_image_paths))

random.seed(42) 
random.shuffle(all_image_paths)

3143


In [7]:
# all_image_paths = all_image_paths[:10000] 

num_total = len(all_image_paths)
num_train = int(num_total * train_ratio)
num_eval = int(num_total * eval_ratio)

splits = {
    "train": all_image_paths[:num_train],
    "eval": all_image_paths[num_train : num_train + num_eval],
    "test": all_image_paths[num_train + num_eval:],
}

for split_name, file_list in splits.items():    
    target_dir = os.path.join(output_dir, split_name, "target")
    source_dir = os.path.join(output_dir, split_name, "source")
    os.makedirs(target_dir, exist_ok=True)
    os.makedirs(source_dir, exist_ok=True)
    
    should_augment = (split_name == "train")
    
    for img_path in tqdm(file_list, desc=f"Processing {split_name}"):
        photo_original = cv2.imread(img_path)
        if photo_original is None:
            continue
        
        photo_resized = cv2.resize(photo_original, (img_size, img_size), interpolation=cv2.INTER_AREA)
        
        sketch_hed = apply_hed(photo_resized, net, target_size=img_size, apply_augmentation=should_augment)
        
        filename = os.path.basename(img_path)
        
        target_save_path = os.path.join(target_dir, filename)
        source_save_path = os.path.join(source_dir, filename)
        
        cv2.imwrite(target_save_path, photo_resized)
        cv2.imwrite(source_save_path, sketch_hed)

Processing train:   0%|          | 0/2828 [00:00<?, ?it/s]

Processing eval:   0%|          | 0/157 [00:00<?, ?it/s]

Processing test:   0%|          | 0/158 [00:00<?, ?it/s]

In [8]:
!zip -r -q /kaggle/working/ffhq_hed_augmented_dataset.zip /kaggle/working/processed_ffhq_hed_augmented